# 🧠 Fine-Tuning de Modelos LLM con LoRA / Hugging Face en Gubernator

Este notebook interactivo demuestra cómo afinar (*fine-tune*) un modelo de lenguaje Open-Source (**SmolLM-135M / Qwen2.5 / TinyLlama**) usando **LoRA (Low-Rank Adaptation)** y la librería **TRL (SFTTrainer)** sobre el almacenamiento distribuido de Gubernator (`/var/contenedores`).

---

### 1. Instalación y Verificación de Dependencias

In [ ]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

### 2. Definición del Dataset de Dominio (Gubernator DevOps Knowledge)

In [ ]:
from datasets import Dataset

raw_data = [
    {
        "prompt": "¿Qué es Gubernator y cuáles son sus puntos fuertes frente a otros orquestadores?",
        "response": "Gubernator es un orquestador 'Goldilocks' que combina la simplicidad de Docker Swarm con el control por tareas de Nomad. Utiliza SQLite distribuido, Caddy Ingress multihost automático, almacenamiento distribuido con GlusterFS (/var/contenedores), observabilidad con Loki/Prometheus/Grafana y control de acceso RBAC con Active Directory/LDAP."
    },
    {
        "prompt": "¿Cómo se despliega un stack en Gubernator?",
        "response": "Ejecuta: `gbnt stack deploy -c docker-compose.yml mi-app`. Gubernator analiza las restricciones de placement, ubica los contenedores en los nodos óptimos y configura automáticamente el enrutamiento Ingress en Caddy y DNS en CoreDNS."
    },
    {
        "prompt": "¿Cómo se hace un backup transaccional de un volumen en Gubernator?",
        "response": "Usa `gbnt backup create --name backup-db --source /var/contenedores/db --pause`. La bandera --pause congela temporalmente las escrituras para máxima consistencia y genera un archivo .tar.gz con checksum SHA-256."
    }
]

# Formatear en estructura de ChatML / Instruct
formatted = [{"text": f"<|im_start|>user\n{d['prompt']}<|im_end|>\n<|im_start|>assistant\n{d['response']}<|im_end|>"} for d in raw_data]
dataset = Dataset.from_list(formatted)
print(f"Total ejemplos: {len(dataset)}")
print("Muestra:", dataset[0]['text'])

### 3. Cargar Tokenizador y Modelo Base

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "HuggingFaceTB/SmolLM-135M-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Cargando {MODEL_ID} en {device}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32 if device == 'cpu' else torch.float16,
    trust_remote_code=True
)

### 4. Configurar Adaptador LoRA (PEFT)

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "v_proj"]
)

peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()

### 5. Entrenamiento con SFTTrainer

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

OUTPUT_DIR = "/home/jovyan/work/output/gubernator-lora-adapter"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    learning_rate=2e-4,
    logging_steps=1,
    optim="adamw_torch",
    report_to="none",
    use_cpu=(device == "cpu")
)

trainer = SFTTrainer(
    model=peft_model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args
)

print("Iniciando entrenamiento...")
trainer.train()

### 6. Guardar Pesos Entrenados y Test de Inferencia

In [ ]:
# Guardar LoRA en disco persistente
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Modelo guardado en {OUTPUT_DIR}")

# Probar inferencia
prompt = "<|im_start|>user\n¿Qué comando uso en Gubernator para hacer un backup seguro?<|im_end|>\n<|im_start|>assistant\n"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = trainer.model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

print("\n--- Respuesta Generada por el Modelo Afinado ---")
print(tokenizer.decode(outputs[0], skip_special_tokens=False))